In [0]:
#import statements
from pyspark.sql.functions import month, count, sum, avg

In [0]:
#copy claims silver table
df_claims = spark.table('insurance_db.silver.claims')

In [0]:
#copy policies silver table
df_policy = spark.table('insurance_db.silver.policies')

In [0]:
#join claims and policies copies via policy_id
joined = df_claims.join(df_policy, 'policy_id', 'left')

In [0]:
#aggregate data by month, count, sum, avg
gold = joined.groupBy(month('claim_date').alias('month'), 'policy_type')\
     .agg(count('claim_id').alias('total_claims'),
          sum('claim_amount').alias('total_amount'),
          avg('claim_amount').alias('avg_amount'))

In [0]:
#make new gold table with the info
gold.write.format('delta').saveAsTable('insurance_db.gold.monthly_claim_summary')

In [0]:
#show the gold table
spark.sql("SELECT * FROM insurance_db.gold.monthly_claim_summary ORDER BY month").show()

+-----+-----------+------------+------------+----------+
|month|policy_type|total_claims|total_amount|avg_amount|
+-----+-----------+------------+------------+----------+
|    1|       AUTO|           1|       250.0|     250.0|
|    2|       AUTO|           1|       200.0|     200.0|
|    3|       HOME|           1|       500.0|     500.0|
|    4|       LIFE|           1|       300.0|     300.0|
|    5|       AUTO|           1|       150.0|     150.0|
|    6|       HOME|           1|       700.0|     700.0|
|    9|       HOME|           1|       600.0|     600.0|
|   10|       LIFE|           1|       350.0|     350.0|
+-----+-----------+------------+------------+----------+

